# PS S6E6 — CatBoost v3

**Baseline LB:** 0.96576 (v2)  
**Why CatBoost:**
- Handles `spectral_type` and `galaxy_population` natively — no ordinal encoding needed
- Different gradient boosting implementation → different errors from LGBM → good ensemble partner
- Often strong out-of-the-box on datasets with categoricals

Same v2 features. Saves `oof_proba_catboost.csv` and `test_proba_catboost.csv` for v5 ensemble.

## 1. Imports & Config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from catboost import CatBoostClassifier

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import balanced_accuracy_score, classification_report

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
CFG = dict(
    n_folds        = 5,
    seed           = 42,
    optuna_trials  = 50,
    optuna_folds   = 3,
    optuna_sample  = 0.3,
    optuna_iters   = 600,
    n_iterations   = 3000,
    early_stop     = 50,
)

BEST_LB = 0.96576

## 2. Load Data

In [ ]:
train = pd.read_csv('/kaggle/input/datasets/ekowannanindome/stellar-dataset/train.csv', index_col='id')
test  = pd.read_csv('/kaggle/input/datasets/ekowannanindome/stellar-dataset/test.csv',  index_col='id')
print(f'Train: {train.shape}  |  Test: {test.shape}')

## 3. Feature Engineering

In [ ]:
def engineer_features(df):
    df = df.copy()
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    df['u_r'] = df['u'] - df['r']
    df['g_i'] = df['g'] - df['i']
    df['g_z'] = df['g'] - df['z']
    df['u_z']          = df['u'] - df['z']
    df['log_redshift'] = np.log1p(df['redshift'])
    df['redshift_sq']  = df['redshift'] ** 2
    df['rs_x_gr']      = df['redshift'] * df['g_r']
    df['rs_x_ug']      = df['redshift'] * df['u_g']
    df['rs_x_gi']      = df['redshift'] * df['g_i']
    return df

train = engineer_features(train)
test  = engineer_features(test)

# CatBoost takes cat features as strings — no encoding needed
CAT_COLS  = ['spectral_type', 'galaxy_population']
NUM_COLS  = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
ENG_COLS  = ['u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'g_i', 'g_z',
             'u_z', 'log_redshift', 'redshift_sq', 'rs_x_gr', 'rs_x_ug', 'rs_x_gi']
FEAT_COLS = NUM_COLS + ENG_COLS + CAT_COLS
TARGET    = 'class'

# Cat feature indices for CatBoost
cat_feature_indices = [FEAT_COLS.index(c) for c in CAT_COLS]

# Encode target
le = LabelEncoder()
y  = le.fit_transform(train[TARGET])
print('Classes:', dict(zip(le.classes_, le.transform(le.classes_))))

X      = train[FEAT_COLS]
X_test = test[FEAT_COLS]
print(f'Feature matrix: {X.shape}')
print(f'Cat feature indices: {cat_feature_indices}')

## 4. Optuna Search

In [ ]:
_, X_opt, _, y_opt = train_test_split(
    X, y, test_size=CFG['optuna_sample'], stratify=y, random_state=CFG['seed']
)
print(f'Optuna search set: {X_opt.shape}')

def objective(trial):
    params = dict(
        iterations        = CFG['optuna_iters'],
        learning_rate     = trial.suggest_float('learning_rate', 0.03, 0.2, log=True),
        depth             = trial.suggest_int('depth', 4, 10),
        l2_leaf_reg       = trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        bagging_temperature = trial.suggest_float('bagging_temperature', 0.0, 1.0),
        random_strength   = trial.suggest_float('random_strength', 0.0, 2.0),
        border_count      = trial.suggest_int('border_count', 32, 255),
        auto_class_weights = 'Balanced',
        eval_metric       = 'TotalF1',
        random_seed       = CFG['seed'],
        verbose           = 0,
        early_stopping_rounds = 30,
        cat_features      = cat_feature_indices,
    )
    skf = StratifiedKFold(n_splits=CFG['optuna_folds'], shuffle=True, random_state=CFG['seed'])
    scores = []
    for tr_idx, val_idx in skf.split(X_opt, y_opt):
        m = CatBoostClassifier(**params)
        m.fit(
            X_opt.iloc[tr_idx], y_opt[tr_idx],
            eval_set=(X_opt.iloc[val_idx], y_opt[val_idx]),
        )
        preds = le.inverse_transform(m.predict(X_opt.iloc[val_idx]).flatten().astype(int))
        truth = le.inverse_transform(y_opt[val_idx])
        scores.append(balanced_accuracy_score(truth, preds))
    return np.mean(scores)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=CFG['seed']))
study.optimize(objective, n_trials=CFG['optuna_trials'], show_progress_bar=True)

print(f'\nBest trial score: {study.best_value:.5f}')
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

## 5. Final 5-Fold CV

In [ ]:
BEST_PARAMS = dict(
    **study.best_params,
    iterations         = CFG['n_iterations'],
    auto_class_weights = 'Balanced',
    eval_metric        = 'TotalF1',
    random_seed        = CFG['seed'],
    verbose            = 200,
    early_stopping_rounds = CFG['early_stop'],
    cat_features       = cat_feature_indices,
)

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
oof_proba  = np.zeros((len(X), len(le.classes_)))
test_proba = np.zeros((len(X_test), len(le.classes_)))
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = CatBoostClassifier(**BEST_PARAMS)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    val_proba = model.predict_proba(X_val)
    val_preds = le.inverse_transform(val_proba.argmax(axis=1))
    score = balanced_accuracy_score(le.inverse_transform(y_val), val_preds)
    fold_scores.append(score)
    oof_proba[val_idx] = val_proba
    test_proba += model.predict_proba(X_test) / CFG['n_folds']

    print(f'  Fold {fold+1} | best_iter={model.best_iteration_} | balanced_acc={score:.5f}')

print(f'\nCV mean: {np.mean(fold_scores):.5f} | std: {np.std(fold_scores):.5f}')

## 6. OOF Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

oof_labels  = le.inverse_transform(oof_proba.argmax(axis=1))
true_labels = le.inverse_transform(y)
oof_score   = balanced_accuracy_score(true_labels, oof_labels)
classes_ordered = ['GALAXY', 'QSO', 'STAR']

print(f'OOF: {oof_score:.5f}  (best LB so far: {BEST_LB})')
print()
print(classification_report(true_labels, oof_labels, target_names=classes_ordered))

ConfusionMatrixDisplay(
    confusion_matrix(true_labels, oof_labels, labels=classes_ordered),
    display_labels=classes_ordered
).plot(cmap='Greens')
plt.title(f'CatBoost v3 OOF — {oof_score:.4f}')
plt.tight_layout()

## 7. Feature Importance

In [ ]:
imp = pd.DataFrame({
    'feature':    FEAT_COLS,
    'importance': model.get_feature_importance(),
}).sort_values('importance', ascending=False)

imp.plot.barh(x='feature', y='importance', figsize=(10, 10), legend=False)
plt.gca().invert_yaxis()
plt.title('CatBoost v3 Feature Importance')
plt.tight_layout()

print(imp.to_string(index=False))

## 8. Save Probabilities + Submission

In [ ]:
pd.DataFrame(oof_proba,  columns=le.classes_).to_csv('oof_proba_catboost.csv',  index=False)
pd.DataFrame(test_proba, columns=le.classes_).to_csv('test_proba_catboost.csv', index=False)

test_pred_labels = le.inverse_transform(test_proba.argmax(axis=1))
submission = pd.DataFrame({'id': test.index, 'class': test_pred_labels})
submission.to_csv('submission_catboost_v3.csv', index=False)

print(f'Submission shape: {submission.shape}')
print(submission['class'].value_counts())

In [ ]:
from IPython.display import FileLink, display
display(FileLink('submission_catboost_v3.csv'))

## 9. Results

| Model | OOF | LB | Notes |
|---|---|---|---|
| LightGBM v2 | 0.96514 | 0.96576 | |
| CatBoost v3 | ... | ... | native categoricals, Balanced weights |

**STAR precision (LGBM was 0.88):**  
**Is CatBoost diverse from LGBM? (different confusion pattern):**